# Week 5: Function Approximation

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab investigates function approximation, a method for representing value estimates that scales and generalizes better than the tabular methods used in our previous labs. Tabular methods store an estimate for each individual state or state-action pair, which works well when the state space is discrete and small. For continuous state spaces the number of possible states is infinite, making tabulation impossible. In addition, a tabular approach exactly maps states or state-action pairs to estimates, which makes generalization impossible.

Function approximation overcomes these constraints by discretizing the continuous state space into regions, and assigning weights to each region. Generalization is further aided by using multiple overlapping discretizations, slightly offset, creating a kind of anti-aliasing effect at the region boundaries. Estimates are therefore blended into a pseudo-continuous approximation. The tradeoff is resolution versus generalization: regions that are too large blur important distinctions, while regions that are too small reduce the sharing of experience between nearby states.

We will demonstrate the concept using the Gymnasium "MountainCar-v0" environment, where state is represented in two dimensions: position, with a continuous range (-1.2, 0.6), and velocity, with a continuous range (-0.07, 0.07). There are three discrete actions: push left, push right, and coast. The goal is reached when position is >= 0.5. Episodes truncate after 200 steps, or terminate when the goal is reached. The reward structure is simple: each step incurs a penalty of -1.

We will implement function approximation for value prediction using a method called "semi-gradient Sarsa," an on-policy approach described in Sutton and Barto, chapters 9 and 10, using ε-greedy exploration for action selection. Here, Sarsa, which is a one-step ahead temporal difference (TD) algorithm, is described as semi-gradient because the mechanism for estimation uses the same weights in the prediction and the target. Instead of deriving the learning gradient from stable values, we are using targets that change as we make predictions. This complicates finding the gradient, and so we don't factor this change into the calculation... we treat the target as stable and apply the update to the prediction. That's why they call it "semi-gradient."

A variety of approaches can be used to implement the value function; this demonstration uses a linear function for regression. The linear regression uses a weight vector that is tuned as the agent interacts with the environment. A binary feature vector derived from the state masks the weight vector, ensuring that the appropriate weights are updated for a given state.

The mechanism for feature construction is tile coding. This approach lays multiple overlapping grids, or "tilings", over the continuous state space, each one slightly offset. For a given state, each tiling identifies which tile the state falls into. The combination of active tiles across all tilings forms the feature vector.

We will experiment with the number of tilings, tile sizes, and offsets to determine their effects. I predict that there is an optimal configuration balancing resolution and generalization, and that deviations in either direction will hurt learning. Too few or too coarse tilings will lack the resolution to distinguish important states, while too many or too fine tilings will reduce generalization and require more experience to learn effectively within a fixed episode budget.

## Section 2: Deliverables

### 2.1 GitHub Repository

GitHub repository: https://github.com/craig-rudman/MSDS684/tree/main/W5

### 2.2. Implementation Summary

This lab demonstrates semi-gradient Sarsa with tile coding using the Gymnasium MountainCar-v0 environment.  Five experiments are conducted in which an agent is trained for 2,500 episodes with various tile configurations using an ε-greedy policy for action selection. The variance in tile configuration translates to a prorated change in α, the learning rate.  All experiments are seeded with the same value (42).

| Label | ε   | α   | α_eff    | Episodes | Tilings | Tiles/Dim | Features |
|-------|-----|-----|----------|----------|---------|-----------|----------|
| Ex1   | 0.1 | 0.1 | 0.01250  | 2500     | 8       | 8×8       | 512      |
| Ex2   | 0.1 | 0.1 | 0.02500  | 2500     | 4       | 8×8       | 256      |
| Ex3   | 0.1 | 0.1 | 0.00625  | 2500     | 16      | 8×8       | 1024     |
| Ex4   | 0.1 | 0.1 | 0.01250  | 2500     | 8       | 4×4       | 128      |
| Ex5   | 0.1 | 0.1 | 0.01250  | 2500     | 8       | 16×16     | 2048     |

*Table 1.  Hyperparameters used in Mountain Car training experiments.*

The experiments vary tiling count and tile resolution to explore how feature design affects learning speed and final performance, with results suggesting 8×8 tiles balance representational capacity and sample efficiency within the 2500-episode budget.

### 2.3 Key Results and Analysis

#### 2.3.1 Did the Agents Learn?

Each of the five agents learned a sequence of actions that would build enough momentum for the car to overcome inertia and reach the goal, but some learned better than others.  Table 2 describes the success rate of each of the agents, and the distribution of results across successful episodes, measured in number of steps.

| Experiment | Success Rate (%) | Mean | Std | Median | Q25 | Q75 | Min | Max |
|------------|-----------------|------|-----|--------|-----|-----|-----|-----|
| Ex1 (8t,8x8)| 90.6 | 153.6 | 12.6 | 154.0 | 148.0 | 159.0 | 95 | 199 |
| Ex2 (4t,8x8)| 87.0 | 153.3 | 8.7 | 152.0 | 148.0 | 158.0 | 97 | 198 |
| Ex3 (16t,8x8)| 91.7 | 150.8 | 19.5 | 155.0 | 149.0 | 161.0 | 91 | 199 |
| Ex4 (8t,4x4)| 82.8 | 159.4 | 10.2 | 157.0 | 152.0 | 164.0 | 114 | 198 |
| Ex5 (8t,16x16)| 66.2 | 154.7 | 11.8 | 155.0 | 150.8 | 160.0 | 91 | 199 |

*Table 2. Episode length statistics for successful episodes (steps < 200) across all five experiments. Ex3 (16 tilings, 8×8) achieves the highest success rate (91.7%) but also the highest variance (std=19.5), suggesting that additional tilings improve coverage at the cost of consistency. Ex5 (8 tilings, 16×16) has the lowest success rate (66.2%), reflecting the difficulty of generalizing from sparse features within a 2500-episode budget. Ex4 (8 tilings, 4×4) solves episodes in the most steps on average (mean=159.4), consistent with its coarser value estimates.*

#### 2.3.2 Learning Speed and Feature Resolution

Figure 1 shows how each agent's success rate evolved over 2,500 episodes. Learning speed varies considerably across tile configurations, with coarser tiles generalizing more broadly and crossing the 50% threshold sooner.

<img src="../img/learning_rates.png">

*Figure 1. Rolling success rate (125-episode window) by feature configuration. The dotted line marks the 50% threshold. Ex4 (4×4) crosses first at episode 111; Ex1, Ex2, and Ex3 (all 8×8) cluster between 233 and 281; Ex5 (16×16) does not cross until episode 798, reflecting the sample cost of fine-grained features.*

Sutton and Barto explain that "features with large receptive fields give broad generalization, but might also seem to limit the learned function to a coarse approximation, unable to make discriminations much finer than the width of the receptive fields" (p. 216). This tradeoff is visible in the results: Ex5's fine-grained 16×16 tiles require far more samples to generalize, yielding the lowest final success rate (66.2%), while Ex4's coarse 4×4 tiles generalize quickly but plateau at 82.8%, also below the 8×8 configurations, consistent with its reduced discriminative capacity. Whether Ex5 would close the gap given a larger episode budget remains an open question; its fine-grained features may simply require more samples to saturate the weight vector, rather than being fundamentally less capable.

Training cost varied primarily with tiling count rather than tile resolution: Ex2 (4 tilings) completed in 26.2 seconds while Ex3 (16 tilings) required 74.3 seconds, nearly three times as long, for a gain of only 4.7 percentage points in success rate (87.0% vs. 91.7%) and higher variance among successful episodes (std=8.7 vs. 19.5). Tile resolution had negligible effect on wall time (Ex4, Ex1, and Ex5 all trained in 43-45 seconds). Table 3 summarizes training cost and solution quality across all five experiments.

| Label | Tilings | Tiles/Dim | Features | Success Rate (%) | Wall Time (s) | Success/Sec | Mean Steps | Std  |
|-------|---------|-----------|----------|-----------------|---------------|-------------|------------|------|
| Ex1   | 8       | 8×8       | 512      | 90.6            | 42.8          | 2.12        | 153.6      | 12.6 |
| Ex2   | 4       | 8×8       | 256      | 87.0            | 26.2          | 3.32        | 153.3      | 8.7  |
| Ex3   | 16      | 8×8       | 1024     | 91.7            | 74.3          | 1.23        | 150.8      | 19.5 |
| Ex4   | 8       | 4×4       | 128      | 82.8            | 44.9          | 1.84        | 159.4      | 10.2 |
| Ex5   | 8       | 16×16     | 2048     | 66.2            | 45.4          | 1.46        | 154.7      | 11.8 |

*Table 3. Training cost and solution quality across all five experiments. Mean Steps and Std are computed from successful episodes only (steps < 200).*



#### 2.3.3 What the Agent Learned

Figure 2 shows the value function agent Ex1 learned after 2,500 episodes. Because the reward is -1 per step, value estimates directly reflect expected steps to goal. The geometry of the heatmap encodes the physics of the problem: low value at the valley floor where momentum is absent, high value at the edges of the state space where velocity can be leveraged to reach the goal directly or via the left hill.

<img src="../img/value_function_plot.png">

*Figure 2. Learned value function (max Q(s,a)) for agent Ex1 after 2,500 training episodes, shown as a heatmap over the state space (position × velocity). The dashed vertical line marks the goal at position=0.5. The red hot spot near (-0.5, 0.0) corresponds to the valley floor with zero momentum, the state requiring the most steps to escape. Green regions at high velocities on either side reflect states from which the agent can reach the goal efficiently, either directly or by first summiting the left hill to build return momentum.*

Given a starting state, the agent is able to navigate to the goal following its learned greedy policy. Figures 3.1 and 3.2 show the trajectory the Ex1 agent takes from five starting positions spaced evenly across the valid position range, each with velocity=0. The trajectories reveal that the agent has learned when to push and when to let gravity assist, building enough velocity on the left hill to summit the right. This behavior emerges directly from the value function: the agent is following the gradient of learned Q-values toward states from which the goal is reachable.

<img src="../img/trajectories_thumbnails.png">
<img src="../img/trajectories_overlay.png">

*Figures 3.1 and 3.2. Greedy policy trajectories for agent Ex1 from five starting positions (position evenly spaced from -1.20 to 0.40, velocity=0), overlaid on the learned value function heatmap. Figure 3.1 (top) shows each trajectory individually; Figure 3.2 (bottom) shows all five together for direct comparison of path length and shape. In each case the agent builds momentum by oscillating toward the left hill before driving rightward to the goal, with trajectories from more favorable starting positions (pos=0.40) requiring fewer oscillations than those starting deeper in the valley (pos=-1.20).*

Figure 4 shows the greedy action the agent selects across the state space. On the left hillside and when climbing the right hill with sufficient velocity, the agent pushes right. When on the right hillside without enough rightward velocity to reach the goal, it pushes left to rebuild momentum. In between, the agent coasts, allowing gravity to assist, primarily when traveling leftward on the right hillside.

<img src="../img/learned_policy_plot.png">

*Figure 4. Learned greedy policy for agent Ex1, showing the action selected at each state (position × velocity). Blue regions indicate push right; orange regions indicate push left; the agent coasts (no push) in the intervening region, primarily when traveling leftward on the right hillside. The policy reflects the momentum-building strategy visible in the trajectory plots: push in the direction of travel to build velocity, and let gravity assist when sufficient momentum has been established.*

#### 2.3.4 Synthesis

MountainCar's state space is described by two continuous dimensions, position and velocity, making the number of possible states virtually infinite and tabular methods intractable. Function approximation takes the view that a policy assigned to subregions of the state space may be good enough to optimize returns. Tile coding implements this by discretizing the continuous space into multiple grids, each offset from the others, so that the active features across tilings overlap in a way that provides finer resolution than would a single grid. The results confirm this: all five agents learned effective policies, with feature design choices determining how quickly and how well generalization occurred.

The experiments reveal a clean separation of effects: tile resolution is the primary driver of outcome quality, with 8×8 tiles producing the highest success rates across configurations, while tiling count is the primary driver of training cost, scaling wall time nearly linearly (26.2s, 42.8s, 74.3s for 4, 8, and 16 tilings respectively) with modest gains in success rate.

By success rate alone, Ex3 is the best model (91.7%); by training efficiency and solution stability, Ex2 is the stronger choice (3.32 successes/sec, std=8.7). Whether the marginal gain in success rate justifies the additional training cost and variance depends on the application: Ex2's 4-tiling configuration trained in 26 seconds with the tightest solution distribution, while Ex3's 16-tiling configuration took nearly three times as long and produced the highest variance (std=19.5) for a modest improvement in success rate, suggesting Ex2 as the more practical choice under a fixed compute budget.

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I asked Claude to help me understand the conceptual foundations of semi-gradient Sarsa with tile coding for the MountainCar environment. Rather than asking for code, my initial prompts focused on understanding the algorithm: what "semi-gradient" means, how tile coding constructs features from continuous states, and how weights produce Q-value estimates. This initial round prepared me for designing the implementation.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration 1

In my first iteration, I worked with Claude to build a MountainCarEnvironment facade over Gymnasium using a test-first approach. My tests caught a bug immediately: I had swapped the observation_space and action_space properties in my implementation. Claude identified this from the red/green output and the fix was straightforward. We initially used a factory pattern to construct the environment, but decided it was unnecessary, opting for a singleton instead.

#### 3.2.2 Iteration 2

In my second iteration, I worked with Claude to implement the TileCoder class. We evaluated whether to expose both get_tiles and get_features, ultimately dropping get_features since the agent can index weight vectors directly using the active indices.

#### 3.2.3 Iteration 3

In my third iteration, I worked with Claude to implement the SarsaAgent class. We evaluated the design carefully before coding: Claude proposed storing weights as a (n_actions, num_features) array, exposing both greedy_action and select_action so the episode runner can use greedy evaluation separately from epsilon-greedy training. After reviewing the instructor's reference, I asked Claude to prorate the learning rate as alpha / n_tilings, avoiding instability from updating 8 tiles simultaneously. Claude also refactored the update method to declare next_q separately, making the Bellman equation more legible.

#### 3.2.4 Subsequent Iterations

In iterations 4 and beyond, we continued building out the training infrastructure using the same test-first discipline established in the first three iterations. We designed and implemented EpisodeRunner and Trainer as separate classes, with Trainer delegating episode execution to EpisodeRunner. Along the way we added agent persistence and an experiment registry to the backlog, anticipating the needs of the feature design experiments. With the training loop in place, the remaining work follows naturally: recording trajectories, visualizing the value function and learned policy, running experiments across configurations, and writing the analysis.

### 3.3 Critical Evaluation

Working in small iterations with test-first development and frequent inspection kept errors to a minimum. Those errors that were made were errors of intent, not coding errors. For example, I identified that the learning rate needed to be prorated across tilings; without this, updating 8 tiles simultaneously would cause instability. I also caught a flaw in the initial experimental design: training runs that terminated at plateau produced unequal episode counts, making comparisons murky. In both cases, the fix was to hold all variables constant except those under investigation. Correctness was verified through the test suite and visual inspection at the end of each iteration for expected learning behavior.

### 3.4 Learning Reflection

Code walkthroughs with Claude were valuable for validating intent and catching legibility issues. Through one such review, I recognized that the TileCoder's binary feature vector was simply a mask of active indices, which led to updating weights by direct index rather than a full dot product. Claude rarely makes outright coding errors, but the code can be dense; making changes for legibility helps ensure correctness. Next time, I would discuss evaluation metrics upfront to avoid rework late in the process.

## Section 4: Speaker Notes

- **Problem:** MountainCar's continuous state space (position x velocity) makes tabular methods intractable; function approximation is required to generalize across infinite possible states.
- **Method:** Implemented semi-gradient SARSA with tile coding, discretizing the continuous state space into multiple offset grids to construct feature vectors for linear Q-value estimation.
- **Design choice:** Defined a three-dimensional evaluation framework (success rate, training cost, solution stability) to compare configurations rather than optimizing for a single metric.
- **Key result:** Ex3 (16t, 8×8) achieved the highest success rate (91.7%) at the highest cost (74.3s); Ex2 (4t, 8×8) offered the best efficiency (3.32 successes/sec) with the tightest solution distribution (std=8.7).
- **Insight:** Tile resolution is the primary driver of outcome quality; tiling count is the primary driver of training cost, scaling wall time nearly linearly with modest gains in success rate.
- **Challenge:** The most significant difficulty was experimental design, specifically holding all variables constant except those under investigation to avoid confounding interpretation of results.
